<a href="https://colab.research.google.com/github/Ape108/FundamentalAnalysisGPT/blob/main/Milestone_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U "datasets<3.0.0"

### Imports

In [ ]:
import re # regular expressions
import math # math utilities
import random # randomly seeding
from collections import Counter # count token frequency
from collections import defaultdict
import string
import datasets

import torch # PyTorch tensor library
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader # dataset and batching

import tiktoken # OpenAI/GPT-style tokenizer

### Seeding For Reproducibility

In [ ]:
# seeding random numbers for reproducibility
random.seed(42)
torch.manual_seed(42)

### Load Corpus

In [ ]:
year_1993_training_dataset = datasets.load_dataset(
    "eloukas/edgar-corpus",
    "year_1993",
    split="train",
    trust_remote_code=True
)

### Dataloader

In [ ]:
dataloader = torch.utils.data.DataLoader(year_1993_training_dataset, batch_size=32)

num_epochs = 1
section_1s = []
for epoch in range(num_epochs):
    for batch_idx, features in enumerate(dataloader): # Iterate through each batch
        print(batch_idx, features['section_1'])
        section_1s.append(features['section_1'])


In [ ]:
text_sample = section_1s[0][0]

## Regex Tokenization

### Define Tokenizer

In [ ]:
def regex_tokenize(text: str) -> list[str]:
    tokens = re.split(r"([,.:;?_!/\"()'\-]|\s+)", text)
    tokens = [t.strip() for t in tokens if t.strip()]
    return tokens

### Tokenize Text and Report

In [ ]:
tokens = regex_tokenize(text_sample)

num_tokens = len(tokens)
print(f"{num_tokens} tokens")
num_unique = len(set(tokens))
print(f"{num_unique} unique tokens")

freq_map = defaultdict(int)
punctuation_count = 0
word_like_count = 0

for tok in tokens:
    freq_map[tok] += 1
    if tok in string.punctuation:
        punctuation_count += 1
    if tok.isalpha(): # If a token contains only characters, consider it word-like
        word_like_count += 1

c = Counter(freq_map) # Referenced Gemini here for Counter syntax
most_common_tokens = dict(c.most_common(30))

print(f"Top 30 tokens by frequency: {most_common_tokens}")


## Vocabulary + Encode/Decode + Special Tokens

### Vocabulary

In [ ]:
def build_vocabulary(tokens):
    unique = sorted(set(tokens)) # get unique tokens and sorting for IDs
    vocab = {tok: i for i, tok in enumerate(unique)} # mapping each token to an ID
    inverse_vocab = {i: tok for tok, i in vocab.items()} # inverse map from ID back to token
    return vocab, inverse_vocab

vocab, inv_vocab = build_vocabulary(tokens)
print(vocab)
print(inv_vocab)
print(f"Vocabulary Size: {len(vocab)}")

### Add Special Tokens

In [ ]:
# Define some common special tokens
SPECIAL_TOKENS = ['<|unk|>', '<|endoftext|>', '[BOS]', '[EOS]', '[PAD]']

def add_special_tokens(vocab, inverse_vocab, special_tokens=SPECIAL_TOKENS):
    vocab = dict(vocab) # Copy vocab so we don't mutate the original
    inverse_vocab = dict(inverse_vocab) # Copy inverse vocab as well
    for tok in special_tokens: # Iterate through the special tokens
        if tok not in vocab: # Only add if it is not already present
            new_token_id = len(vocab) # Assign the next available ID
            vocab[tok] = new_token_id # Add token-->id mapping
            inverse_vocab[new_token_id] = tok # Add id-->token mapping
    return vocab, inverse_vocab

vocab2, inverse_vocab2 = add_special_tokens(vocab, inv_vocab, SPECIAL_TOKENS)
print('New vocab size:', len(vocab2))
print(vocab2)


### Implement Encoder & Decoder

In [ ]:
def encode_with_unk(text, vocab, unk_token='<|unk|>'):
    """Convert a list of tokens into token IDs"""
    toks = regex_tokenize(text) # Tokenize the new input text
    unk_id = vocab[unk_token] # Get the ID for the unknown tokens
    ids = [vocab.get(t, unk_id) for t in toks] # We use vocab.get to fall back to unk_id
    return ids, toks

def decode(ids, inverse_vocab):
    """Convert a list of token IDs back to tokens (readable strings)"""
    toks = [inverse_vocab[i] for i in ids] # map each ID to its token
    text = ' '.join(toks) # join tokens with spaces
    text = re.sub(r"\s+([,.:;?_!/\"()'])", r"\1", text)
    text = re.sub(r"\s+\-", "-", text)
    return text


In [ ]:
ids, toks = encode_with_unk(text_sample, vocab2)
dec = decode(ids, inverse_vocab2)
print("Text Sample:")
print(dec[:120] + '...')
with open("corpus_sample.txt", 'w') as file:
    file.write(text_sample)


# Tokenizer Write-Up

For the SEC EDGAR corpus, I have implemented a Regular-Expression tokenizer that splits tokens by white space and punctuation. It intetntionally preserves capitalization, given its semanting importance in corporate filings ("Apple" vs "apple"). The vocabulary is built dynamically from the training corpus, where the length is the exact number of unique tokens in the corpus. It also includes "<|unk|>" and other special tokens to handle special scenarios. While this approach avoids fragmenting complicated financial terms, it also loses the computational efficiency of subword tokenization by producing a larger embedding matrix and relying on "<|unk|>" strictly for out-of-vocab words.